In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, RepeatedKFold
import pickle
import torch
import pickle
import re
from collections.abc import Generator, Iterable
from dataclasses import dataclass
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from chemprop.data import MoleculeDatapoint, MoleculeDataset
from chemprop.featurizers import MultiHotAtomFeaturizer, MultiHotBondFeaturizer, SimpleMoleculeMolGraphFeaturizer
from rdkit import Chem
from rdkit.Chem.rdchem import HybridizationType
from scipy.constants import (
    Avogadro,  # 1/mol
    Boltzmann,  # in J/K
)
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from ml_enhance import QuantumFPFileLoader, parallelize

In [3]:
with open("all_features.pkl", "rb") as f:
    data = pickle.load(f)

In [4]:
target_df = pd.read_csv("../data/processed_dataset_rerun.csv")[["smiles", "id", "solubility"]]

In [5]:
target_df["id"] = target_df["id"].apply(round)
target_df.to_csv("target_df.csv")

In [6]:
def build_nested_cv_splits(
    target_df,
    outer_cv,
    inner_cv,
    id_col="id",
):
    ids = target_df[id_col].to_numpy()

    nested_splits = {}

    for outer_i, (outer_tr_idx, outer_te_idx) in enumerate(outer_cv.split(ids)):
        outer_train_ids = ids[outer_tr_idx]
        outer_test_ids = ids[outer_te_idx]

        inner_folds = []

        for inner_tr_idx, inner_val_idx in inner_cv.split(outer_train_ids):
            inner_train_ids = outer_train_ids[inner_tr_idx]
            inner_val_ids = outer_train_ids[inner_val_idx]

            inner_folds.append(
                {
                    "train_ids": inner_train_ids,
                    "val_ids": inner_val_ids,
                }
            )

        nested_splits[f"outer_fold_{outer_i}"] = {
            "train_ids": outer_train_ids,
            "test_ids": outer_test_ids,
            "inner_folds": inner_folds,
        }

    return nested_splits

In [20]:
outer_cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=15)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

nested_splits = build_nested_cv_splits(target_df, outer_cv, inner_cv)

In [19]:
pd.DataFrame(nested_splits) #.to_csv("chemprop_splits.csv")

,outer_fold_0,outer_fold_1,outer_fold_2,outer_fold_3,outer_fold_4,outer_fold_5,outer_fold_6,outer_fold_7,outer_fold_8,outer_fold_9,...,outer_fold_15,outer_fold_16,outer_fold_17,outer_fold_18,outer_fold_19,outer_fold_20,outer_fold_21,outer_fold_22,outer_fold_23,outer_fold_24
train_ids,"[0, 1, 10, 100, 1000, 1002, 1003, 1004, 1007, ...","[0, 1, 100, 1001, 1002, 1003, 1004, 1005, 1008...","[10, 1000, 1001, 1003, 1005, 1007, 1008, 1009,...","[0, 1, 10, 100, 1000, 1001, 1002, 1003, 1004, ...","[0, 1, 10, 100, 1000, 1001, 1002, 1004, 1005, ...","[1, 10, 1000, 1001, 1002, 1003, 1005, 1007, 10...","[0, 1, 10, 100, 1000, 1001, 1002, 1004, 1005, ...","[0, 1, 10, 100, 1002, 1003, 1004, 1005, 1007, ...","[0, 1, 100, 1000, 1001, 1002, 1003, 1004, 1005...","[0, 10, 100, 1000, 1001, 1003, 1004, 1007, 101...",...,"[0, 1, 10, 100, 1000, 1001, 1002, 1008, 1009, ...","[0, 1, 10, 100, 1001, 1002, 1003, 1004, 1005, ...","[10, 1000, 1003, 1004, 1005, 1007, 1008, 1009,...","[0, 1, 10, 100, 1000, 1001, 1002, 1003, 1004, ...","[0, 1, 100, 1000, 1001, 1002, 1003, 1004, 1005...","[0, 10, 100, 1001, 1002, 1005, 1007, 1008, 100...","[1, 10, 100, 1000, 1002, 1003, 1004, 1005, 100...","[0, 1, 10, 100, 1000, 1001, 1002, 1003, 1004, ...","[0, 1, 1000, 1001, 1002, 1003, 1004, 1005, 100...","[0, 1, 10, 100, 1000, 1001, 1003, 1004, 1005, ..."
test_ids,"[1001, 1005, 1008, 1012, 1017, 1018, 1022, 102...","[10, 1000, 1007, 1011, 1014, 1025, 1026, 1027,...","[0, 1, 100, 1002, 1004, 102, 1020, 1032, 1033,...","[1013, 1021, 1024, 1029, 1031, 1037, 1038, 103...","[1003, 1009, 1010, 1015, 1016, 1019, 103, 1035...","[0, 100, 1004, 1021, 1026, 1034, 1042, 1048, 1...","[1003, 1010, 1013, 1016, 1017, 1019, 1032, 103...","[1000, 1001, 1012, 1014, 1015, 1018, 1028, 102...","[10, 1007, 1011, 1020, 1022, 1025, 103, 1041, ...","[1, 1002, 1005, 1008, 1009, 102, 1023, 1024, 1...",...,"[1003, 1004, 1005, 1007, 1010, 1021, 1024, 102...","[1000, 1008, 1014, 1018, 102, 1022, 1025, 1039...","[0, 1, 100, 1001, 1002, 1016, 1017, 1019, 1020...","[1011, 1015, 1023, 1028, 1029, 1036, 1041, 105...","[10, 1009, 1012, 1013, 1032, 1043, 1045, 105, ...","[1, 1000, 1003, 1004, 1011, 1014, 1016, 1019, ...","[0, 1001, 1018, 1027, 103, 1032, 1033, 1044, 1...","[1005, 1010, 1015, 1022, 1024, 1031, 1034, 103...","[10, 100, 1017, 1025, 1026, 1037, 1046, 1051, ...","[1002, 1007, 1008, 1009, 1012, 1013, 102, 1021..."
inner_folds,"[{'train_ids': [0, 1, 10, 100, 1000, 1002, 100...","[{'train_ids': [0, 1, 100, 1001, 1002, 1003, 1...","[{'train_ids': [10, 1000, 1001, 1003, 1005, 10...","[{'train_ids': [0, 1, 10, 100, 1000, 1001, 100...","[{'train_ids': [0, 1, 10, 100, 1000, 1001, 100...","[{'train_ids': [1, 10, 1000, 1001, 1002, 1003,...","[{'train_ids': [0, 1, 10, 100, 1000, 1001, 100...","[{'train_ids': [0, 1, 10, 100, 1002, 1003, 100...","[{'train_ids': [0, 1, 100, 1000, 1001, 1002, 1...","[{'train_ids': [0, 10, 100, 1000, 1001, 1003, ...",...,"[{'train_ids': [0, 1, 10, 100, 1000, 1001, 100...","[{'train_ids': [0, 1, 10, 100, 1001, 1002, 100...","[{'train_ids': [10, 1000, 1003, 1004, 1005, 10...","[{'train_ids': [0, 1, 10, 100, 1000, 1001, 100...","[{'train_ids': [0, 1, 100, 1000, 1001, 1002, 1...","[{'train_ids': [0, 10, 100, 1001, 1002, 1005, ...","[{'train_ids': [1, 10, 100, 1000, 1002, 1003, ...","[{'train_ids': [0, 1, 10, 100, 1000, 1001, 100...","[{'train_ids': [0, 1, 1000, 1001, 1002, 1003, ...","[{'train_ids': [0, 1, 10, 100, 1000, 1001, 100..."


In [18]:
pd.read_csv("chemprop_splits.csv").to_dict()

{'Unnamed: 0': {0: 'train_ids', 1: 'test_ids', 2: 'inner_folds'},
 'outer_fold_0': {0: '[  0   1  10 ... 997 998 999]',
  1: '[1001 1005 1008 ...  976  978  982]',
  2: "[{'train_ids': array([  0,   1,  10, ..., 995, 996, 999], shape=(5605,)), 'val_ids': array([1007, 1015, 1016, ...,  991,  997,  998], shape=(1402,))}, {'train_ids': array([  0,   1,  10, ..., 997, 998, 999], shape=(5605,)), 'val_ids': array([1013,  103, 1033, ...,  992,  994,  995], shape=(1402,))}, {'train_ids': array([  1,  10, 100, ..., 995, 997, 998], shape=(5606,)), 'val_ids': array([   0, 1003, 1004, ...,  993,  996,  999], shape=(1401,))}, {'train_ids': array([   0,  100, 1000, ...,  997,  998,  999], shape=(5606,)), 'val_ids': array([   1,   10, 1010, ...,  983,  987,  989], shape=(1401,))}, {'train_ids': array([  0,   1,  10, ..., 997, 998, 999], shape=(5606,)), 'val_ids': array([ 100, 1000, 1002, ...,  984,  986,  990], shape=(1401,))}]"},
 'outer_fold_1': {0: '[  0   1 100 ... 997 998 999]',
  1: '[  10 1000

In [15]:
pd.__version__

'3.0.1'

In [16]:
import pickle
with open("chemprop_splits.pkl", "wb") as f:
    pickle.dump(nested_splits, f)

In [17]:
with open("chemprop_splits.pkl", "rb") as f:
    data = pickle.load(f)

data

{'outer_fold_0': {'train_ids': array([  0,   1,  10, ..., 997, 998, 999], shape=(7007,)),
  'test_ids': array([1001, 1005, 1008, ...,  976,  978,  982], shape=(1752,)),
  'inner_folds': [{'train_ids': array([  0,   1,  10, ..., 995, 996, 999], shape=(5605,)),
    'val_ids': array([1007, 1015, 1016, ...,  991,  997,  998], shape=(1402,))},
   {'train_ids': array([  0,   1,  10, ..., 997, 998, 999], shape=(5605,)),
    'val_ids': array([1013,  103, 1033, ...,  992,  994,  995], shape=(1402,))},
   {'train_ids': array([  1,  10, 100, ..., 995, 997, 998], shape=(5606,)),
    'val_ids': array([   0, 1003, 1004, ...,  993,  996,  999], shape=(1401,))},
   {'train_ids': array([   0,  100, 1000, ...,  997,  998,  999], shape=(5606,)),
    'val_ids': array([   1,   10, 1010, ...,  983,  987,  989], shape=(1401,))},
   {'train_ids': array([  0,   1,  10, ..., 997, 998, 999], shape=(5606,)),
    'val_ids': array([ 100, 1000, 1002, ...,  984,  986,  990], shape=(1401,))}]},
 'outer_fold_1': {'trai

In [ ]:
def build_lookup(df, group_col, feature_cols):
    if df is None:
        return None
    
    return (
        df.groupby(group_col)[feature_cols]
        .apply(lambda x: x.to_numpy(dtype=float).copy())
        .to_dict()
    )

In [82]:
def split_df_by_ids(df, ids, id_col="id"):
    return df.set_index(id_col).loc[ids].reset_index()

In [83]:
def index_features(features):
    indexed = {}
    for key, df in features.items():
        if df is not None:
            indexed[key] = df.set_index("original_smiles")
        else:
            indexed[key] = None
    return indexed

def subset_features(
    features: dict[str, pd.DataFrame | None],
    smiles: list[str],
) -> dict[str, pd.DataFrame | None]:
    return {
        key: df[df["original_smiles"].isin(smiles)].reset_index(drop=True) if df is not None else None
        for key, df in features.items()
    }

In [84]:
def make_datapoints(
    target_df: pd.DataFrame,
    atom_lookup: dict[str, np.ndarray] | None,
    bond_lookup: dict[str, np.ndarray] | None,
    mol_lookup: dict[str, np.ndarray] | None,
    target_col: str = "solubility",
) -> list[MoleculeDatapoint]:
    datapoints = []

    for row in target_df.itertuples(index=False):
        smiles = row.smiles
        y = np.array([getattr(row, target_col)], dtype=float)

        V_f = atom_lookup.get(smiles) if atom_lookup is not None else None
        E_f = bond_lookup.get(smiles) if bond_lookup is not None else None
        x_d = mol_lookup.get(smiles) if mol_lookup is not None else None

        datapoints.append(MoleculeDatapoint.from_smi(smi=smiles, y=y, V_f=V_f, E_f=E_f, x_d=x_d, keep_h=True))

    return datapoints

In [ ]:
from data_preprocess import scale_features, apply_rbf, bond_features, atomic_features, mol_features, scale_target, get_featurizer, Config

In [ ]:
def build_datasets(
    train_ids: np.ndarray,
    val_ids: np.ndarray,
    target_df: pd.DataFrame,
    all_features: dict[str, pd.DataFrame | None],
    *,
    config: Config,
) -> tuple[MoleculeDataset, MoleculeDataset, StandardScaler]:

    train_target_df = split_df_by_ids(target_df, train_ids)
    val_target_df   = split_df_by_ids(target_df, val_ids)

    train_smiles = train_target_df["smiles"].tolist()
    val_smiles = val_target_df["smiles"].tolist()

    train_features = subset_features(all_features, train_smiles)
    val_features = subset_features(all_features, val_smiles)

    train_atom_df = train_features["atoms"] if config.use_atom_features else None
    val_atom_df = val_features["atoms"] if config.use_atom_features else None

    train_bond_df = train_features["bonds"] if config.use_bond_features else None
    val_bond_df = val_features["bonds"] if config.use_bond_features else None

    train_mol_df = train_features["mols"] if config.use_mol_features else None
    val_mol_df = val_features["mols"] if config.use_mol_features else None

    atom_rbf_cols: list[str] = []
    bond_rbf_cols: list[str] = []

    if (train_atom_df is not None) and (val_atom_df is not None):
        train_atom_df, val_atom_df, _ = scale_features(train_atom_df, val_atom_df, atomic_features)
        train_atom_df, atom_rbf_cols = apply_rbf(train_atom_df, atomic_features, config.n_rbf_centers)
        val_atom_df, _ = apply_rbf(val_atom_df, atomic_features, config.n_rbf_centers)

    if (train_bond_df is not None) and (val_bond_df is not None):
        train_bond_df, val_bond_df, _ = scale_features(train_bond_df, val_bond_df, bond_features)
        train_bond_df, bond_rbf_cols = apply_rbf(train_bond_df, bond_features, config.n_rbf_centers)
        val_bond_df, _ = apply_rbf(val_bond_df, bond_features, config.n_rbf_centers)

    if (train_mol_df is not None) and (val_mol_df is not None):
        train_mol_df, val_mol_df, _ = scale_features(train_mol_df, val_mol_df, mol_features)

    train_target_df, val_target_df, target_scaler = scale_target(train_target_df, val_target_df, target_col)


    train_atom_lookup = build_lookup(train_atom_df, "original_smiles", atom_rbf_cols) if train_atom_df is not None else None
    val_atom_lookup   = build_lookup(val_atom_df, "original_smiles", atom_rbf_cols) if val_atom_df is not None else None

    train_bond_lookup = build_lookup(train_bond_df, "original_smiles", bond_rbf_cols) if train_bond_df is not None else None
    val_bond_lookup   = build_lookup(val_bond_df, "original_smiles", bond_rbf_cols) if val_bond_df is not None else None

    train_mol_lookup = (
        None if train_mol_df is None else
        train_mol_df.groupby("original_smiles")[mol_features]
        .apply(lambda x: np.atleast_1d(x.to_numpy(dtype=np.float32).squeeze()))
        .to_dict()
    )

    val_mol_lookup = (
        None if val_mol_df is None else
        val_mol_df.groupby("original_smiles")[mol_features]
        .apply(lambda x: np.atleast_1d(x.to_numpy(dtype=np.float32).squeeze()))
        .to_dict()
    )


    train_datapoints = make_datapoints(
        train_target_df,
        train_atom_lookup,
        train_bond_lookup,
        train_mol_lookup,
        config.target_col,
    )

    val_datapoints = make_datapoints(
        val_target_df,
        val_atom_lookup,
        val_bond_lookup,
        val_mol_lookup,
        config.target_col,
    )

    featurizer = get_featurizer(
        use_custom_atom_featurizer=config.use_custom_atom_featurizer,
        use_custom_bond_featurizer=config.use_custom_bond_featurizer,
        extra_atom_fdim=len(atom_rbf_cols) if train_atom_df is not None else 0,
        extra_bond_fdim=len(bond_rbf_cols) if train_bond_df is not None else 0,
    )
    return (
        MoleculeDataset(train_datapoints, featurizer=featurizer),
        MoleculeDataset(val_datapoints, featurizer=featurizer),
        target_scaler,
    )

In [87]:
result = build_datasets(nested_splits["outer_fold_0"]["train_ids"], nested_splits["outer_fold_0"]["test_ids"], target_df, data)

In [88]:
nested_splits["outer_fold_0"]["test_ids"][0]

np.int64(1001)

In [92]:
target_df[target_df["id"] == 1001].smiles.values[0]

'[C:1](/[C:2](=[C:3](\\[C:4]([H:22])([H:23])[H:24])[C:5](=[O:6])[O:7][C:8]([C:9]([C:10]([C:11]([H:30])([H:31])[H:32])([C:12]([C:13]([C:14](=[C:15]([C:16]([H:38])([H:39])[H:40])[C:17]([H:41])([H:42])[H:43])[H:37])([H:35])[H:36])([H:33])[H:34])[H:29])([H:27])[H:28])([H:25])[H:26])[H:21])([H:18])([H:19])[H:20]'

In [98]:
mol = Chem.MolFromSmiles('[C:1](/[C:2](=[C:3](\\[C:4]([H:22])([H:23])[H:24])[C:5](=[O:6])[O:7][C:8]([C:9]([C:10]([C:11]([H:30])([H:31])[H:32])([C:12]([C:13]([C:14](=[C:15]([C:16]([H:38])([H:39])[H:40])[C:17]([H:41])([H:42])[H:43])[H:37])([H:35])[H:36])([H:33])[H:34])[H:29])([H:27])[H:28])([H:25])[H:26])[H:21])([H:18])([H:19])[H:20]')
mol.GetNumBonds()

16

In [103]:
result[1][0].mg._asdict()

{'V': array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         2.03822411e-01, 1.33448245e-02, 2.30310900e-04],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         2.97675347e-02, 7.15660404e-04, 4.53536037e-06],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         3.55824605e-02, 9.25532044e-04, 6.34581833e-06],
        ...,
        [1.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         8.00986322e-02, 3.07094125e-03, 3.10354901e-05],
        [1.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         7.21346218e-02, 2.62213726e-03, 2.51250986e-05],
        [1.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         7.86968072e-02, 2.98999123e-03, 2.99449372e-05]], shape=(43, 162)),
 'E': array([[0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        ...,
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 